# Multi-Agent Demo (Autogen style) - Template

**But :** prototype pédagogique montrant la coordination de plusieurs agents. Ce notebook offre deux variantes :

- **Variante A (mock)** : agents simulés en Python (threads / fonctions) qui échangent des messages.
- **Variante B (Autogen)** : instructions et template pour utiliser Microsoft Autogen si tu veux installer la librairie.

Le notebook logge les échanges et produit un diagramme simple des interactions.

## Installation (terminal)
```bash
pip install networkx matplotlib nbformat
# Optionnel pour Autogen (vérifier compatibilité):
# pip install autogen
```

In [ ]:
import time
import json
from collections import deque

print('env ready')

## Variante A : Simulation simple multi-agent
Trois agents : `researcher`, `planner`, `executor`. Ils échangent des messages via une queue centralisée. Chaque agent a un rôle et peut demander l'avis des autres.

In [ ]:
class SimpleAgent:
    def __init__(self, name, role):
        self.name = name
        self.role = role
        self.inbox = deque()
    def send(self, other, message):
        other.inbox.append({'from':self.name, 'message':message})
    def step(self):
        out = []
        while self.inbox:
            msg = self.inbox.popleft()
            print(f"[{self.name}] received from {msg['from']}: {msg['message']}")
            # Simple role behavior
            if self.role == 'researcher':
                # returns findings
                out.append({'to':'planner', 'message':f"findings about {msg['message']} : important data"})
            elif self.role == 'planner':
                out.append({'to':'executor', 'message':f"plan based on {msg['message']} : do step 1"})
            elif self.role == 'executor':
                out.append({'to':'user', 'message':f"executed {msg['message']}"})
        return out

# Create agents
researcher = SimpleAgent('researcher','researcher')
planner = SimpleAgent('planner','planner')
executor = SimpleAgent('executor','executor')

# Start scenario
user_req = 'Organiser une demo produit la semaine prochaine'
researcher.inbox.append({'from':'user','message':user_req})

# Run steps
for _ in range(3):
    for ag in [researcher, planner, executor]:
        outs = ag.step()
        for o in outs:
            if o['to'] == 'planner':
                planner.inbox.append({'from':ag.name,'message':o['message']})
            elif o['to'] == 'executor':
                executor.inbox.append({'from':ag.name,'message':o['message']})
            elif o['to'] == 'user':
                print('[USER] ', o['message'])

print('Scenario finished')

## Variante B : Template pour Microsoft Autogen (si tu veux utiliser Autogen)
> Autogen permet de définir agents avec des rôles et des callbacks. Ci‑dessous un pseudo‑exemple (vérifie la doc officielle et installe la lib si besoin).

In [ ]:
# PSEUDO-CODE Autogen (non exécuté ici)
# from autogen import AssistantAgent, UserProxy, GroupChatManager
#
# analyst = AssistantAgent(name='analyst', role='Collect and summarize relevant documents')
# planner = AssistantAgent(name='planner', role='Create plan based on analyst summary')
# executor = AssistantAgent(name='executor', role='Execute plan steps and report')
#
# manager = GroupChatManager(agents=[analyst, planner, executor])
# manager.run('Plan a product demo next week')

print('Autogen pseudo-code provided. See Microsoft Autogen docs to run this locally.')

## Visualisation simple des échanges (graph)
On va tracer un graphe avec networkx montrant qui envoie à qui.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
G = nx.DiGraph()
G.add_edge('user','researcher')
G.add_edge('researcher','planner')
G.add_edge('planner','executor')
G.add_edge('executor','user')

pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_size=2000, node_color='lightblue', arrowsize=20)
plt.title('Simple multi-agent flow')
plt.show()

## Bench simple et métriques
Quelques métriques à collecter :
- nombre d'itérations pour converger
- latence par étape
- cohérence des réponses (requiert évaluation humaine ou heuristique)

Mettre en place un logger, stocker les échanges et calculer ces métriques.

In [ ]:
# Exemple de logging des échanges
logs = []
# re-run small scenario and log
researcher.inbox.append({'from':'user','message':'Verifier disponibilite salle demo'})
for _ in range(3):
    for ag in [researcher, planner, executor]:
        outs = ag.step()
        for o in outs:
            logs.append({'from':ag.name,'to':o['to'],'message':o['message']})
            if o['to']=='planner': planner.inbox.append({'from':ag.name,'message':o['message']})
            if o['to']=='executor': executor.inbox.append({'from':ag.name,'message':o['message']})
            if o['to']=='user': print('[USER]', o['message'])

print('Logs:')
print(json.dumps(logs, indent=2, ensure_ascii=False))

## Packaging du projet multi-agent
- Créer un repo GitHub avec README, instructions pour exécuter
- Ajouter Dockerfiles pour chaque service agent si tu veux déployer en microservices
- Ajouter tests unitaires et scénarios d'intégration

---

Fin du notebook 2.